# FoldSplice evals

Reads `results/<eval>/<model>.{npz,tsv}` written by `foldsplice eval`. Figures use the first read-out of each model (`psi` for site, `psplice` for tissue); the tables list all of them.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from foldsplice.evals import bltp1, compensatory, mapt, pnas
from foldsplice.models import family

RES = Path("../results")
ARMS = ["seq", "fold"]
COLORS = {"measured": "black", "seq": "#9e9e9e", "fold": "#2a6fb0"}


def runs(name):
    return [m for m in ("site", "tissue") if (RES / name / f"{m}.npz").exists()]


def readout(z, arm):
    return next(iter(family(str(z["model"])).readouts(z[arm], 0, 1).values()))


def table(name):
    return pd.concat([pd.read_csv(RES / name / f"{m}.tsv", sep="\t") for m in runs(name)])

## PNAS: calibrated error by exon MFE

A structure-blind model over-predicts inclusion of tightly folded exons (left bins).

In [ ]:
BINS = [-np.inf, -30, -25, -20, -15, -10, np.inf]
LABELS = ["< -30", "(-30, -25]", "(-25, -20]", "(-20, -15]", "(-15, -10]", "-10+"]
fig, axes = plt.subplots(1, len(runs("pnas")), figsize=(5 * len(runs("pnas")), 3.4), squeeze=False)
for ax, m in zip(axes[0], runs("pnas")):
    z = np.load(RES / "pnas" / f"{m}.npz")
    psi, bins = z["psi"] * 100, pd.cut(z["mfe"], BINS, labels=LABELS)
    for k, arm in enumerate(ARMS):
        p = readout(z, arm).mean(0)
        err = pnas.calibrator(p, psi)(p) - psi
        ax.boxplot([err[bins == b] for b in LABELS], positions=np.arange(6) + 0.36 * k - 0.18, widths=0.32,
                   showfliers=False, patch_artist=True, boxprops=dict(facecolor=COLORS[arm]),
                   medianprops=dict(color="black"))
    ax.axhline(0, color="black", lw=0.6)
    ax.set_xticks(range(6), LABELS, rotation=30)
    ax.set(title=m, xlabel="predicted MFE (kcal/mol)", ylabel="calibrated error (PSI points)")
axes[0, 0].legend([plt.Rectangle((0, 0), 1, 1, fc=COLORS[a]) for a in ARMS], ARMS)
plt.tight_layout()
table("pnas")

## Compensatory stem loops

Measured inclusion rises with each stem-breaking single and falls back with the re-pairing double.

In [ ]:
t = compensatory.table()
fig, axes = plt.subplots(len(runs("compensatory")), 5, figsize=(13, 2.6 * len(runs("compensatory"))), squeeze=False, sharey=True)
for row, m in zip(axes, runs("compensatory")):
    z, ref = np.load(RES / "compensatory" / f"{m}.npz"), np.load(RES / "pnas" / f"{m}.npz")
    lines = {"measured": t.psi.to_numpy() * 100}
    for arm in ARMS:
        lines[arm] = pnas.calibrator(readout(ref, arm).mean(0), ref["psi"] * 100)(readout(z, arm).mean(0))
    for k, ax in enumerate(row):
        for name, y in lines.items():
            y = y.reshape(-1, 4)[k]
            ax.plot(range(4), y - y[0], "o-", color=COLORS[name], label=name)
        ax.axhline(0, color="black", lw=0.6)
        ax.set_xticks(range(4), compensatory.VARIANTS, rotation=30)
        ax.set_title(f"{m} {t.series[4 * k]}")
    row[0].set_ylabel("PSI change from WT")
axes[0, -1].legend()
plt.tight_layout()
table("compensatory")

## MAPT exon 10

Error against the stability of the exon 10 / intron 10 stem (unfolding ΔG‡, lower = more stable).

In [ ]:
v = pd.read_csv(mapt.DATA / "variants.tsv", sep="\t")
fig, axes = plt.subplots(len(runs("mapt")), 2, figsize=(8, 3.2 * len(runs("mapt"))), squeeze=False, sharey=True)
for row, m in zip(axes, runs("mapt")):
    z = np.load(RES / "mapt" / f"{m}.npz")
    for ax, arm in zip(row, ARMS):
        err = (readout(z, arm).mean(0)[1:] - v.psi) * 100
        slope, icpt = np.polyfit(v.struc, err, 1)
        xs = np.linspace(v.struc.min(), v.struc.max(), 2)
        ax.scatter(v.struc, err, s=14, color=COLORS[arm])
        ax.plot(xs, slope * xs + icpt, "k--")
        ax.axhline(0, color="black", lw=0.6)
        ax.set(title=f"{m} {arm}: r = {np.corrcoef(err, v.struc)[0, 1]:+.2f}", xlabel="strucMean (kcal/mol)")
    row[0].set_ylabel("error (PSI points)")
plt.tight_layout()
table("mapt")

## BLTP1 c.13476C>T

Measured ΔPSI is −50 (dashed); dots are replicates.

In [ ]:
fig, axes = plt.subplots(1, len(runs("bltp1")), figsize=(3 * len(runs("bltp1")), 3.2), squeeze=False, sharey=True)
for ax, m in zip(axes[0], runs("bltp1")):
    z = np.load(RES / "bltp1" / f"{m}.npz")
    for k, arm in enumerate(ARMS):
        s = readout(z, arm)
        d = (s[:, 1] - s[:, 0]) * 100
        ax.bar(k, d.mean(), color=COLORS[arm], width=0.6)
        ax.scatter(k + np.linspace(-0.15, 0.15, len(d)), d, s=6, color="black", zorder=3)
    ax.axhline(bltp1.MEASURED_DPSI, color="black", ls="--")
    ax.set_xticks(range(2), ARMS)
    ax.set(title=m, ylabel="ΔPSI (variant − WT)")
plt.tight_layout()
table("bltp1")